# NewsQA fine-tuning with boundary-distillation regularization (anti-forgetting)

This is our main mitigation experiment. It continues fine-tuning the SQuAD-trained BERT model on NewsQA, same as `finetune_bert_on_newsqa.ipynb`, but adds a regularization term during training that keeps the model's start and end logits close to a frozen copy of the original SQuAD-trained model's logits on the same inputs.

This follows the Learning without Forgetting approach (Li and Hoiem, 2016): instead of needing old-task data to prevent forgetting, keep a frozen "teacher" copy of the old model and penalize the new "student" model for drifting too far from the teacher's output distribution while it learns the new task. We apply this specifically to the start/end span logits, which is the architectural angle the team wanted (not a change to the training data).

Compare the results here against:
- `bert_baseline_results.md`: SQuAD-only baseline (EM 80.92 / F1 88.40)
- `newsqa_finetune_results.md`: plain sequential fine-tune, no regularization (SQuAD EM 74.84 / F1 85.44, NewsQA EM 48.55 / F1 64.24)

Requires `train-newsqa.json` / `dev-newsqa.json` (from `preprocess_newsqa.ipynb`) and `qa_model` (from `train_qa.ipynb`) to already exist in Drive.

In [1]:
!pip install -q -U transformers datasets accelerate evaluate

import transformers
print("transformers version:", transformers.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
transformers version: 5.22.1


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import collections
import subprocess
import dataclasses
import inspect
import copy

import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

PROJECT_DIR = "/content/drive/MyDrive/CS4248 Group Project"
DATA_DIR = os.path.join(PROJECT_DIR, "qa")

BASE_MODEL_DIR = os.path.join(PROJECT_DIR, "qa_model")   # SQuAD-trained model, used as both the student's starting point and the frozen teacher

NEWSQA_TRAIN_FILE = os.path.join(DATA_DIR, "train-newsqa.json")
NEWSQA_DEV_FILE = os.path.join(DATA_DIR, "dev-newsqa.json")
SQUAD_DEV_FILE = os.path.join(DATA_DIR, "dev-v1.1.json")
EVAL_SCRIPT = os.path.join(DATA_DIR, "evaluate-v2.0.py")

MAX_LENGTH = 384
DOC_STRIDE = 128
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5       # same as the plain sequential fine-tune, for a fair comparison
NUM_EPOCHS = 2
N_BEST = 20
MAX_ANSWER_LENGTH = 30

# Distillation-specific hyperparameters
DISTILL_WEIGHT = 1.0       # how strongly to penalize drifting from the teacher's logits, relative to the NewsQA task loss
DISTILL_TEMPERATURE = 2.0  # softens the logit distributions before computing KL divergence, standard in distillation

OUTPUT_DIR = os.path.join(PROJECT_DIR, "qa_model_newsqa_distill")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PREDICTIONS_NEWSQA_FILE = os.path.join(DATA_DIR, "predictions_newsqa_distill.json")
PREDICTIONS_SQUAD_RECHECK_FILE = os.path.join(DATA_DIR, "predictions_squad_after_newsqa_distill.json")

MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

In [4]:
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > Hardware "
    "accelerator > GPU (T4), save, reconnect, then re-run from the top."
)
print("GPU:", torch.cuda.get_device_name(0))

for path in (BASE_MODEL_DIR, NEWSQA_TRAIN_FILE, NEWSQA_DEV_FILE, SQUAD_DEV_FILE, EVAL_SCRIPT):
    assert os.path.exists(path), (
        f"Missing: {path}\n"
        "If this is train-newsqa.json / dev-newsqa.json, run preprocess_newsqa.ipynb first.\n"
        "If this is qa_model, make sure train_qa.ipynb has been run and saved there."
    )
print("All required files found.")

GPU: Tesla T4
All required files found.


In [5]:
def load_squad_json(path):
    with open(path) as f:
        squad = json.load(f)
    ids, questions, contexts, answers = [], [], [], []
    for article in squad["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                ids.append(qa["id"])
                questions.append(qa["question"])
                contexts.append(context)
                answers.append(
                    {
                        "text": [a["text"] for a in qa["answers"]],
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                    }
                )
    return Dataset.from_dict(
        {"id": ids, "question": questions, "context": contexts, "answers": answers}
    )


print("Loading NewsQA data...")
raw_datasets = DatasetDict(
    {
        "train": load_squad_json(NEWSQA_TRAIN_FILE),
        "validation": load_squad_json(NEWSQA_DEV_FILE),
    }
)

print("Loading original SQuAD dev set, for the forgetting check at the end...")
squad_dev_raw = load_squad_json(SQUAD_DEV_FILE)

if MAX_TRAIN_SAMPLES is not None:
    raw_datasets["train"] = raw_datasets["train"].select(range(MAX_TRAIN_SAMPLES))
if MAX_EVAL_SAMPLES is not None:
    raw_datasets["validation"] = raw_datasets["validation"].select(range(MAX_EVAL_SAMPLES))

raw_datasets

Loading NewsQA data...
Loading original SQuAD dev set, for the forgetting check at the end...


DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers'],
        num_rows: 10003
    })
    validation: Dataset({
        features: ['id', 'question', 'context', 'answers'],
        num_rows: 1003
    })
})

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
assert tokenizer.is_fast, "Need a fast tokenizer for offset mapping"

In [7]:
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = tokenized.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


def prepare_validation_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    example_ids = []
    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    tokenized["example_id"] = example_ids
    return tokenized


print("Tokenizing NewsQA train set...")
train_dataset = raw_datasets["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

print("Tokenizing NewsQA validation set...")
validation_dataset = raw_datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

print("Tokenizing original SQuAD dev set (for the forgetting check)...")
squad_validation_dataset = squad_dev_raw.map(
    prepare_validation_features,
    batched=True,
    remove_columns=squad_dev_raw.column_names,
)

Tokenizing NewsQA train set...


Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Tokenizing NewsQA validation set...


Map:   0%|          | 0/1003 [00:00<?, ? examples/s]

Tokenizing original SQuAD dev set (for the forgetting check)...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

## Student and frozen teacher

Both start from the same SQuAD-trained checkpoint. The student is fine-tuned on NewsQA; the teacher is frozen and only used to produce soft targets for the regularization term.

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

student_model = AutoModelForQuestionAnswering.from_pretrained(BASE_MODEL_DIR).to(device)

teacher_model = AutoModelForQuestionAnswering.from_pretrained(BASE_MODEL_DIR).to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad_(False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
class DistillationTrainer(Trainer):
    """Trainer that adds a Learning-without-Forgetting style regularizer:
    penalizes the student's start/end logits for drifting from a frozen
    teacher's logits on the same (NewsQA) inputs, alongside the normal
    NewsQA span-prediction loss.
    """

    def __init__(self, *args, teacher_model=None, distill_weight=1.0, distill_temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.distill_weight = distill_weight
        self.distill_temperature = distill_temperature

    def _kl(self, student_logits, teacher_logits):
        t = self.distill_temperature
        student_log_probs = F.log_softmax(student_logits / t, dim=-1)
        teacher_probs = F.softmax(teacher_logits / t, dim=-1)
        return F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (t ** 2)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        task_loss = outputs.loss

        teacher_inputs = {k: v for k, v in inputs.items() if k not in ("start_positions", "end_positions")}
        with torch.no_grad():
            teacher_outputs = self.teacher_model(**teacher_inputs)

        distill_loss = (
            self._kl(outputs.start_logits, teacher_outputs.start_logits)
            + self._kl(outputs.end_logits, teacher_outputs.end_logits)
        ) / 2

        loss = task_loss + self.distill_weight * distill_loss
        return (loss, outputs) if return_outputs else loss

In [10]:
ta_fields = {f.name for f in dataclasses.fields(TrainingArguments)}


def pick(*name_value_pairs):
    for name, value in name_value_pairs:
        if name in ta_fields:
            return {name: value}
    return {}


ta_kwargs = {"output_dir": OUTPUT_DIR}
ta_kwargs.update(pick(("eval_strategy", "no"), ("evaluation_strategy", "no")))
ta_kwargs.update(pick(("save_strategy", "epoch")))
ta_kwargs["learning_rate"] = LEARNING_RATE
ta_kwargs["per_device_train_batch_size"] = BATCH_SIZE
ta_kwargs["per_device_eval_batch_size"] = BATCH_SIZE
ta_kwargs["gradient_accumulation_steps"] = GRADIENT_ACCUMULATION_STEPS
ta_kwargs["num_train_epochs"] = NUM_EPOCHS
ta_kwargs["weight_decay"] = 0.01
ta_kwargs.update(pick(("warmup_ratio", 0.1)))
ta_kwargs.update(pick(("fp16", torch.cuda.is_available())))
ta_kwargs.update(pick(("gradient_checkpointing", True)))
ta_kwargs.update(pick(("dataloader_num_workers", 0)))
ta_kwargs.update(pick(("report_to", "none")))
ta_kwargs.update(pick(("run_name", "bert-newsqa-distill")))

args = TrainingArguments(**ta_kwargs)

trainer_params = set(inspect.signature(Trainer.__init__).parameters)
tokenizer_kwarg = "processing_class" if "processing_class" in trainer_params else "tokenizer"

trainer = DistillationTrainer(
    model=student_model,
    args=args,
    train_dataset=train_dataset,
    teacher_model=teacher_model,
    distill_weight=DISTILL_WEIGHT,
    distill_temperature=DISTILL_TEMPERATURE,
    **{tokenizer_kwarg: tokenizer},
)

In [11]:
print("Fine-tuning on NewsQA with boundary-distillation regularization...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Fine-tuning on NewsQA with boundary-distillation regularization...


Step,Training Loss
500,2.049992
1000,1.940510
1500,1.849199
2000,1.573792
2500,1.505016
3000,1.518385


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/CS4248 Group Project/qa_model_newsqa_distill


## Evaluate on NewsQA dev and re-check SQuAD dev

Same inference and decoding logic as the other notebooks, so results are directly comparable.

In [12]:
def run_inference(model, dataset, batch_size=BATCH_SIZE):
    model.to(device)
    model.eval()

    eval_set_for_model = dataset.remove_columns(["example_id", "offset_mapping"])
    eval_set_for_model.set_format("torch")

    all_start_logits, all_end_logits = [], []
    with torch.no_grad():
        for i in range(0, len(eval_set_for_model), batch_size):
            batch = eval_set_for_model[i : i + batch_size]
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_start_logits.append(outputs.start_logits.cpu().numpy())
            all_end_logits.append(outputs.end_logits.cpu().numpy())

    return np.concatenate(all_start_logits, axis=0), np.concatenate(all_end_logits, axis=0)


def postprocess_qa_predictions(examples, features, all_start_logits, all_end_logits):
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feature_id]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        context = example["context"]
        feature_indices = features_per_example[example_index]

        valid_answers = []
        for feature_index in feature_indices:
            start_logit = all_start_logits[feature_index]
            end_logit = all_end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -N_BEST - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -N_BEST - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offsets)
                        or end_index >= len(offsets)
                        or offsets[start_index] is None
                        or offsets[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > MAX_ANSWER_LENGTH:
                        continue
                    start_char, end_char = offsets[start_index][0], offsets[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logit[start_index] + end_logit[end_index],
                            "text": context[start_char:end_char],
                        }
                    )

        if valid_answers:
            best = max(valid_answers, key=lambda x: x["score"])
            predictions[example["id"]] = best["text"]
        else:
            predictions[example["id"]] = ""
    return predictions


def score(data_file, predictions_file):
    result = subprocess.run(
        ["python", EVAL_SCRIPT, data_file, predictions_file],
        check=True, capture_output=True, text=True,
    )
    print(result.stdout)

In [13]:
print("Running inference on NewsQA validation set...")
newsqa_start_logits, newsqa_end_logits = run_inference(student_model, validation_dataset)
newsqa_predictions = postprocess_qa_predictions(
    raw_datasets["validation"], validation_dataset, newsqa_start_logits, newsqa_end_logits
)
with open(PREDICTIONS_NEWSQA_FILE, "w") as f:
    json.dump(newsqa_predictions, f)
print("Scoring on NewsQA dev...")
score(NEWSQA_DEV_FILE, PREDICTIONS_NEWSQA_FILE)

Running inference on NewsQA validation set...
Scoring on NewsQA dev...
{
  "exact": 45.66301096709871,
  "f1": 61.47269759374965,
  "total": 1003,
  "HasAns_exact": 45.66301096709871,
  "HasAns_f1": 61.47269759374965,
  "HasAns_total": 1003
}



In [14]:
print("Running inference on original SQuAD dev set...")
squad_start_logits, squad_end_logits = run_inference(student_model, squad_validation_dataset)
squad_predictions = postprocess_qa_predictions(
    squad_dev_raw, squad_validation_dataset, squad_start_logits, squad_end_logits
)
with open(PREDICTIONS_SQUAD_RECHECK_FILE, "w") as f:
    json.dump(squad_predictions, f)
print("Scoring on original SQuAD dev (post distillation fine-tuning)...")
score(SQUAD_DEV_FILE, PREDICTIONS_SQUAD_RECHECK_FILE)

Running inference on original SQuAD dev set...
Scoring on original SQuAD dev (post distillation fine-tuning)...
{
  "exact": 80.07568590350047,
  "f1": 88.05610563310528,
  "total": 10570,
  "HasAns_exact": 80.07568590350047,
  "HasAns_f1": 88.05610563310528,
  "HasAns_total": 10570
}



## What to look for

Compare the SQuAD re-check numbers here against the plain sequential fine-tune (EM 74.84 / F1 85.44, no regularization). If distillation is helping, SQuAD numbers should be closer to the original baseline (EM 80.92 / F1 88.40) while NewsQA numbers stay reasonably close to the plain fine-tune's (EM 48.55 / F1 64.24).

If SQuAD retention improves but NewsQA performance drops a lot, `DISTILL_WEIGHT` is probably too high, the regularizer is dominating over actually learning the new task. Try lowering it (e.g. 0.5 or 0.25) and re-running.